<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip-rnn-bit-parity-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Bit-parity classifier

🎯 **Task Overview**: This notebook builds an RNN to classify bit sequences based on parity:
- Output `1` for sequences with an odd number of `1`s (odd parity)
- Output `0` for sequences with an even number of `1`s (even parity)


# Setup

In [17]:
!pip install wandb tsilva-notebook-utils==0.0.21 > /dev/null

🔑 Loading API keys and authentication tokens from Colab secrets:


In [18]:
from tsilva_notebook_utils.colab import load_secrets_into_env

load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [19]:
import os
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    #@markdown ### 📌 General Settings

    #@markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    #@markdown Number of training epochs
    n_epochs = 10000  # @param {type:"integer"}

    #@markdown Batch size
    batch_size = 32  # @param {type:"integer"}

    #@markdown ### 📈 Optimizer Settings

    #@markdown Learning rate for the optimizer
    learning_rate = 0.0005  # @param {type:"number"}

    #@markdown Weight decay for the optimizer
    weight_decay = 0  # @param {type:"number"}

    #@markdown Learning rate warmup ratio (0.0 to 1.0)
    warmup_ratio = 0  # @param {type:"number"}

    #@markdown Maximum gradient norm for clipping (0.0 means no clipping)
    max_grad_norm = 0  # @param {type:"number"}

    #@markdown ### 🧠 Model Architecture

    #@markdown Number of hidden units in the RNN
    hidden_size = 4  # @param {type:"integer"}

    #@markdown Activation function to use in the RNN
    nonlinearity = 'tanh'  # @param ['tanh', 'sigmoid', 'relu']

    #@markdown Weight initialization strategy
    weight_init = 'xavier'  # @param ['none', 'xavier', 'kaiming']
    if weight_init == 'none': weight_init = None

    #@markdown ### 📊 Data Settings

    #@markdown Length of the input sequence (number of time steps)
    sequence_length = 5  # @param {type:"integer"}

    # Hardcoded internal settings
    train_size = 0.8
    val_size = 0.2
    batch_size = 32  # Overwrites user input
    input_size = 1   # One input, one digit at a time
    output_size = 2  # Two outputs: even or odd

    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'weight_decay': weight_decay,
        'warmup_ratio': warmup_ratio,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size
    }

CONFIG = setup_config()

🔒 Setting seed for reproducible results:


In [20]:
from tsilva_notebook_utils.colab import set_seed
set_seed(CONFIG['seed'])

📊 Generating parity dataset with binary sequences:


In [21]:
import random
import torch
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from tsilva_notebook_utils.torch import get_current_device, inspect_tensor_dataset

def generate_parity_dataset(
    sequence_length=None,
    val_size=None,
    seed=None
):
    if sequence_length is None: sequence_length = CONFIG['sequence_length']
    if val_size is None: val_size = CONFIG['val_size']
    if seed is None: seed = CONFIG['seed']

    # Set seed for reproducibility
    set_seed(seed)

    data = []
    labels = []

    # Generate all binary sequences for each length in the range
    sequences = list(itertools.product([0, 1], repeat=sequence_length))  # All combinations of 0s and 1s
    for seq in sequences:
        parity = sum(seq) % 2  # Label is 1 if number of 1s is odd, else 0
        data.append(seq)
        labels.append(parity)

    # Convert data and labels to PyTorch tensors on the correct device
    device = get_current_device()
    X = torch.tensor(data, dtype=torch.float32, device=device).unsqueeze(-1)  # Shape: (N, sequence_length, 1)
    Y = torch.tensor(labels, dtype=torch.long, device=device)

    # Shuffle the dataset
    perm = torch.randperm(len(X))
    X = X[perm]
    Y = Y[perm]

    # Split into training and validation datasets
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=val_size, random_state=seed, shuffle=True
    )

    # Create TensorDatasets
    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)

    return train_dataset, val_dataset

train_dataset, val_dataset = generate_parity_dataset()
inspect_tensor_dataset(train_dataset)

{'num_samples': 25,
 'tensor_shapes': [torch.Size([25, 5, 1]), torch.Size([25])],
 'sample_data': [(tensor([[0.],
           [1.],
           [0.],
           [1.],
           [1.]], device='cuda:0'),
   tensor(1, device='cuda:0')),
  (tensor([[0.],
           [1.],
           [0.],
           [1.],
           [0.]], device='cuda:0'),
   tensor(0, device='cuda:0')),
  (tensor([[0.],
           [0.],
           [1.],
           [1.],
           [0.]], device='cuda:0'),
   tensor(0, device='cuda:0'))]}

⚙️ Creating DataLoaders for efficient batch processing:


In [22]:
from torch.utils.data import DataLoader

# Create DataLoader
batch_size = CONFIG['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Sample a batch (iterator)
X_batch, Y_batch = next(iter(train_loader))
X_batch.shape, Y_batch.shape

(torch.Size([25, 5, 1]), torch.Size([25]))

🧠 Building the RNN model with customizable architecture:


In [23]:
import torch.nn as nn
from tsilva_notebook_utils.torch import apply_weight_init

class ParityRNN(nn.Module):
    def __init__(
        self,
        input_size=None,
        hidden_size=None,
        output_size=None,
        nonlinearity=None,
        weight_init=None
    ):
        super(ParityRNN, self).__init__()

        if input_size is None: input_size = CONFIG['input_size']
        if hidden_size is None: hidden_size = CONFIG['hidden_size']
        if output_size is None: output_size = CONFIG['output_size']
        if nonlinearity is None: nonlinearity = CONFIG['nonlinearity']
        if weight_init is None: weight_init = CONFIG['weight_init']

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            batch_first=True,
            nonlinearity=nonlinearity
        )

        self.fc = nn.Linear(hidden_size, output_size)

        if weight_init: apply_weight_init(self, weight_init, nonlinearity)

    def forward(self, x):
        # Pass the input sequence through the RNN layer
        # hidden_states contains the output features from the RNN at each time step
        # last_hidden_state is the final hidden state for the last time step
        hidden_states, last_hidden_state = self.rnn(x)

        # The final hidden state is squeezed to remove the first dimension (num_layers)
        # This is required because RNN returns output in shape: (num_layers * num_directions, batch, hidden_size)
        # Since we're using a single-layer, single-direction RNN, we can safely squeeze it to (batch, hidden_size)
        logits = self.fc(last_hidden_state[0]) # NOTE: the first dimension is for selecting layer/direction, since this model only has 1 layer and 1 direction, there is only index 0

        # logits represent the raw output scores from the final fully connected layer
        # These can be passed through a softmax or sigmoid depending on your classification task
        return logits, hidden_states

device = get_current_device()
model = ParityRNN().to(device)
logits, _ = model(X_batch)
logits

tensor([[-0.1190,  3.1641],
        [-0.0431,  3.2332],
        [-0.8847,  3.0842],
        [-0.8361,  3.1650],
        [-0.0688,  3.2894],
        [-2.1120,  0.2351],
        [-2.3871,  0.1809],
        [-1.0697,  2.7248],
        [-1.2227,  2.5100],
        [-2.0019,  0.5929],
        [-2.6394,  0.1972],
        [-0.3732,  3.3109],
        [-1.1305,  2.6009],
        [-2.0657,  0.5889],
        [-0.0336,  0.7107],
        [-0.2763,  3.3677],
        [ 0.2522,  1.0040],
        [-0.2098,  3.4928],
        [ 1.6783,  1.8615],
        [ 1.0546,  0.2847],
        [-0.2389,  3.3534],
        [ 0.0000,  0.0000],
        [-2.2101,  0.2702],
        [ 0.9092,  0.9101],
        [-0.8215,  3.1835]], device='cuda:0', grad_fn=<AddmmBackward0>)

📝 Evaluating model performance before training:


In [24]:
loss_fn = nn.CrossEntropyLoss()

def _eval(loader, collect_misses=False):
    losses, accuracies, misses = [], [], []
    for x_batch, y_batch in loader:
        # Forward pass
        with torch.no_grad(): logits, _ = model(x_batch)

        # Calculate loss
        loss = loss_fn(logits, y_batch)
        losses.append(loss.item())

        # Calculate accuracy
        predicted = torch.argmax(logits, dim=-1)
        accuracy = (predicted == y_batch).float().mean().item()
        accuracies.append(accuracy)

        # Continue in case we're not collecting misses
        if not collect_misses:
            continue

        # Continue if there are no mismatches
        mismatches = predicted != y_batch
        if not mismatches.any():
            continue

        # Extract the misclassified inputs and their true/predicted labels
        misses_x = x_batch[mismatches]
        misses_y = y_batch[mismatches]
        misses_y_pred = predicted[mismatches]

        # Collect the details of each misclassified example
        for x, y, y_pred in zip(misses_x, misses_y, misses_y_pred):
            misses.append({
                'x': x.cpu(),
                'y': y.item(),
                'y_pred': y_pred.item()
            })

    # Compute average loss and accuracy over all batches
    avg_loss = sum(losses) / len(loader)
    avg_accuracy = sum(accuracies) / len(accuracies)
    results = {
        'loss': avg_loss,
        'accuracy': avg_accuracy
    }
    if collect_misses: results['misses'] = misses
    return results

def eval(loader, collect_misses=False):
    model.eval()  # Set model to evaluation mode (e.g., disables dropout, batchnorm tracking)
    try: return _eval(loader, collect_misses=collect_misses)  # Run evaluation
    finally: model.train()  # Revert model to training mode

{
    "train": eval(train_loader),
    "val": eval(val_loader)
}

{'train': {'loss': 1.3388537168502808, 'accuracy': 0.5600000023841858},
 'val': {'loss': 1.843698501586914, 'accuracy': 0.2857142984867096}}

📈 Initializing Weights & Biases (wandb) for experiment tracking:


In [25]:
from tsilva_notebook_utils.wandb import init_with_defaults
init_with_defaults(CONFIG)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


🚀 Training the model with early stopping at perfect accuracy:


In [26]:
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
import wandb
from tsilva_notebook_utils.torch import calc_model_grad_norms

def train(
    n_epochs=None,
    learning_rate=None,
    max_grad_norm=None,
    weight_decay=None,
    warmup_ratio=None
):
    if n_epochs is None: n_epochs = CONFIG['n_epochs']
    if learning_rate is None: learning_rate = CONFIG['learning_rate']
    if max_grad_norm is None: max_grad_norm = CONFIG['max_grad_norm']
    if weight_decay is None: weight_decay = CONFIG['weight_decay']
    if warmup_ratio is None: warmup_ratio = CONFIG['warmup_ratio']

    # Set model to training mode
    model.train()

    # Watch model for gradient logging with wandb
    wandb.watch(model, log="all")

    # Set up optimizer and learning rate scheduler
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Linear warmup schedule: linearly increase LR for warmup_steps, then keep constant
    total_steps = n_epochs * len(train_loader)
    warmup_steps = int(warmup_ratio * total_steps)
    def lr_lambda(current_step): return float(current_step) / float(max(1, warmup_steps)) if current_step < warmup_steps else 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    # Progress bar for training
    global_step = 0
    with tqdm(range(n_epochs), desc="Training") as pbar:
        for epoch in pbar:
            losses, accuracies, grad_norms, layer_norms = [], [], [], {}
            last_layer_norms = {}

            # Iterate over batches
            for x_batch, y_batch in train_loader:
                global_step += 1

                # Forward pass
                logits, _ = model(x_batch)

                # Calculate loss
                loss = loss_fn(logits, y_batch)
                losses.append(loss.item())

                # Perform backpropagation
                optimizer.zero_grad()
                loss.backward()

                # Optionally clip gradients to avoid exploding gradients
                if max_grad_norm > 0: clip_grad_norm_(model.parameters(), max_grad_norm)

                # Calculate gradient norms for monitoring
                _grad_norm, _layer_norms = calc_model_grad_norms(model)
                grad_norms.append(_grad_norm)

                last_layer_norms = layer_norms

                # Perform optimization step
                optimizer.step()

                # Perform learning rate scheduler step
                scheduler.step()

                # Calculate training accuracy
                with torch.no_grad():
                    predicted = torch.argmax(logits, dim=1)
                    accuracy = (predicted == y_batch).float().mean().item()
                    accuracies.append(accuracy)

            # Run evaluation on validation set
            result = eval(val_loader)
            val_loss = result['loss']
            val_accuracy = result['accuracy']

            # Log W&B stats
            train_loss = sum(losses) / len(losses)
            train_accuracy = sum(accuracies) / len(accuracies)
            train_grad_norm = sum(grad_norms) / len(grad_norms)
            current_lr = scheduler.get_last_lr()[0]
            stats = {
                'meta/epoch': epoch + 1,
                'meta/step': global_step,
                'meta/learning_rate': current_lr,

                'train/loss': train_loss,
                'train/accuracy': train_accuracy,

                'val/loss': val_loss,
                'val/accuracy': val_accuracy,

                'grad/total_norm': train_grad_norm,
            }
            for _layer_name, _layer_norm in _layer_norms.items(): stats[f'grad/layer/{_layer_name}'] = _layer_norm
            wandb.log(stats, step=global_step)

            # Show key metrics in progress bar
            pbar.set_postfix({
                'epoch': epoch + 1,
                'lr': f'{current_lr:.6e}',
                'grad_norm': f'{train_grad_norm:.2f}',
                'train_acc': f'{train_accuracy * 100:.2f}%',
                'val_acc': f'{val_accuracy * 100:.2f}%'
            })

            # Early stopping if training accuracy hits 100%
            if train_accuracy == 1.0:
                print("\nTraining finished early: model perfectly fit the training set.")
                break

    wandb.finish()

train()

Training:  23%|██▎       | 2302/10000 [00:20<01:08, 112.33it/s, epoch=2303, lr=5.000000e-04, grad_norm=0.12, train_acc=100.00%, val_acc=100.00%]



Training finished early: model perfectly fit the training set.


grad/layer/fc.bias,█▇▇▇▆▅▄▄▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▁▁▁▁
grad/layer/fc.weight,█▆▆▅▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▁▁▁▁▁
grad/layer/rnn.bias_hh_l0,█▆▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
grad/layer/rnn.bias_ih_l0,█▅▅▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
grad/layer/rnn.weight_hh_l0,▇███▅▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▂▂▁▁▁▁▁▁▁▁▁
grad/layer/rnn.weight_ih_l0,▇██▇▆▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
grad/total_norm,███▅▅▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
meta/epoch,▁▁▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
meta/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
meta/step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/accuracy,▂▂▂▂▂▁▁▁▁▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█


🔍 Evaluating final model and analyzing missed predictions:


In [27]:
def log_eval_results(results):
    for i, miss in enumerate(results['misses']):
        x = miss["x"]
        x_s = "".join([str(_x.item()) for _x in x.int().squeeze()])
        correct = "odd" if miss["y"] == 1 else "even"
        predicted = "odd" if miss["y_pred"] == 1 else "even"
        print(f"{x_s}: {correct} (predicted: {predicted})")
    print(f"Validation accuracy: {results['accuracy'] * 100:.2f}%")

eval_results = eval(val_loader, collect_misses=True)
log_eval_results(eval_results)

Validation accuracy: 100.00%


🧪 Testing model with manual input sequences:


In [ ]:
def manual_test():
    device = get_current_device()
    while True:
        sequence_s = input()
        if sequence_s == "exit": return
        x = torch.tensor(list(map(int, sequence_s)), device=device).unsqueeze(-1).float()
        with torch.no_grad(): logits, _ = model(x)
        prediction = torch.argmax(logits, dim=0)
        prediction = prediction.item()
        prediction_s = "even" if prediction == 0 else "odd"
        print(prediction_s)

# Uncomment this line to manually test predictions
# (commented by default to allow running `notify_and_disconnect_after_timeout` in next cell)
manual_test()

⏰ Setting up auto-notification and resource management:


In [ ]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()